# GPT multi-label markup (sample 2k)

Loads ~2000 articles from the last 14 days, labels them via OpenAI API.

- Prompt + taxonomy: `src/labeling/prompt.py`
- API runner: `src/labeling/openai_labeler.py`
- Output: `data/labels/article_labels_2k.parquet` (resumable)

Set in `.env`: `OPENAI_API_KEY`, optional `OPENAI_MODEL` (default `gpt-4o-mini`, or e.g. `gpt-4.1-nano`).

In [1]:
import os

from src.config import get_settings
from src.db_utils.pg_utils import select_query
from src.labeling.prompt import THEME_LABELS

get_settings().postgres_dsn
os.getenv("OPENAI_MODEL", "gpt-4o-mini"), THEME_LABELS

('gpt-4o-mini',
 ('politics',
  'economics',
  'technology',
  'conflicts',
  'energy',
  'sanctions',
  'markets',
  'regulation',
  'corporate',
  'crime',
  'disasters',
  'health',
  'science',
  'sports',
  'entertainment',
  'lifestyle',
  'other'))

In [ ]:
sample = select_query("""
    SELECT
        a.article_id,
        a.title,
        a.text,
        a.domain,
        a.datetime,
        a.lang
    FROM raw.news_articles a
    WHERE a.datetime >= '2026-09-30' - interval '1 year'
      AND a.text IS NOT NULL
      AND length(trim(a.text)) > 100
    ORDER BY gen_random_uuid()
    LIMIT 2000
""")

sample.shape

In [5]:
sample.select("title", "domain", "datetime", "lang").head(10)

title,domain,datetime,lang
str,str,"datetime[μs, Etc/UTC]",str
"""Venezuela’s crisis is not an o…","""www.aljazeera.com""",2025-12-11 00:00:00 UTC,"""en"""
"""AFP сообщила, что США требуют …","""www.rbc.ru""",2025-12-15 00:00:00 UTC,"""ru"""
"""Военные сборы — 2025: что нужн…","""www.rbc.ru""",2025-11-07 00:00:00 UTC,"""ru"""
"""Попрошаек брали под охрану""","""www.kommersant.ru""",2025-12-14 00:00:00 UTC,"""ru"""
"""Саакашвили не признал вину по …","""www.kommersant.ru""",2026-02-10 00:00:00 UTC,"""ru"""
"""Россия не уведомляла Казахстан…","""kommersant.ru""",2026-08-19 12:39:47 UTC,"""ru"""
"""Российский Як-130М получит «Пр…","""lenta.ru""",2026-09-29 03:00:21 UTC,"""ru"""
"""Sudan: Quartet Settlement in S…","""allafrica.com""",2026-02-04 00:00:00 UTC,"""en"""
"""Judge clears way for Minnesota…","""www.foxnews.com""",2026-01-07 00:00:00 UTC,"""en"""


In [6]:
from src.labeling.openai_labeler import label_articles

OUTPUT = "data/labels/article_labels_2k.parquet"

labels_df = label_articles(sample, OUTPUT, sleep_s=0.05)
labels_df.head(10)

ModuleNotFoundError: No module named 'openai'

In [ ]:
labels_df.explode("labels").group_by("labels").len().sort("len", descending=True)